# Examples: large model–recording L2 when the emitter swims or moves its tail fast

For each recording and each kind of motion — **swimming speed** (velocity) and **tail angular speed** (tail third relative
to the body axis), both from `EOD_motion_plots_my_pipeline.ipynb` / `eod_motion.py` — 20 good-frame EODs (better list)
where the fit is bad:
1. the emitter is among the fastest 15% of the good-frame EODs for that measure; for tail angular speed, its swimming
   speed is also below that cutoff, so the tail examples are about the tail and not about fast swimming;
2. the fit is among the worst 25%: the larger of the two models' L2 for the emitter — **mine** (numerical, sink at tail tip,
   sign fixed, from `eod_assignments.csv`) and **Denis'** (his normalized predicted vs recorded peak-to-peak, latest run) —
   both on patterns scaled to max |v| = 1;
3. EODs where the nearest other fish is at least `FAR_CM` away (`gap_cm`) first, then the rest, each by decreasing L2;
4. at least `MIN_GAP_FRAMES` video frames apart.

Each figure is `eod_examples.plot_example` (tank, zoom on the emitter, recorded vs model traces per candidate fish).
Figures go to `results/eod_ground_truth{,_3fish}/conductivities/examples_fast_motion/<measure>/`, with the selection in
`selection.csv`.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt                  # figures are saved, not shown (mask_metrics sets the Agg backend)
import eod_examples as ex
warnings.filterwarnings('ignore', message='FigureCanvasAgg is non-interactive')

N_EXAMPLES, FAST_Q, BAD_Q, FAR_CM, MIN_GAP_FRAMES = 20, 0.85, 0.75, 10, 25
KINDS = {'velocity_cm_s': ('fast swimming', 'cm/s'), 'tail_angular_deg_s': ('fast tail movement', '°/s')}
BETTER = {'3-fish': 'results/eod_ground_truth_3fish/good_frames_better_list.csv',
          '2-fish': 'results/eod_ground_truth/good_frames_better_list.csv'}

## Select the examples

In [2]:
def denis_l2(c):
    '''L2 between Denis' normalized predicted pattern of the emitter and the normalized recorded peak-to-peak pattern.'''
    X = {f: x for x, f in c['denis_map'].items()}[c['emitter']]
    return pd.Series(np.linalg.norm(np.stack(c['den'][f'normalized_predicted_PTP_{X}']) - np.stack(c['den']['normalized_recorded_PTP']),
                                    axis=1), index=c['df'].index)

def select(M, measure):
    '''N_EXAMPLES rows of M (good-frame EODs with motion) per the rules above.'''
    cand = M[(M[measure] >= M[measure].quantile(FAST_Q)) & (M.l2_worst >= M.l2_worst.quantile(BAD_Q))]
    if measure != 'velocity_cm_s':                                   # tail examples: not swimming fast
        cand = cand[cand.velocity_cm_s < M.velocity_cm_s.quantile(FAST_Q)]
    cand = cand.assign(far=cand.gap_cm >= FAR_CM).sort_values(['far', 'l2_worst'], ascending=[False, False])
    picked = []
    for i, r in cand.iterrows():
        if all(abs(r.Frame - M.Frame[j]) >= MIN_GAP_FRAMES for j in picked):
            picked.append(i)
        if len(picked) == N_EXAMPLES: break
    return M.loc[picked].assign(far=lambda s: s.gap_cm >= FAR_CM)

recs, selections = {}, {}
for name in ('3-fish', '2-fish'):
    c = ex.emitter_usual_lengths(ex.use_better_list(ex.load_recording(name), BETTER[name]))
    M = pd.read_csv(f"{c['res']}/motion_per_eod_better_list.csv", index_col='row').rename(columns={'l2': 'l2_mine'})
    assert (M.Frame == c['df'].Frame[M.index]).all()
    M['l2_denis'] = denis_l2(c)[M.index]
    M['l2_worst'] = M[['l2_mine', 'l2_denis']].max(axis=1)
    M = M.dropna(subset=list(KINDS))
    recs[name] = c
    for measure in KINDS:
        sel = select(M, measure)
        selections[name, measure] = sel
        print(f"{name}, {measure}: fast ≥ {M[measure].quantile(FAST_Q):.0f}, worst L2 ≥ {M.l2_worst.quantile(BAD_Q):.2f}; "
              f"{len(sel)} picked, {sel.far.sum()} with the other fish ≥ {FAR_CM} cm")

3-fish, velocity_cm_s: fast ≥ 17, worst L2 ≥ 1.01; 20 picked, 20 with the other fish ≥ 10 cm
3-fish, tail_angular_deg_s: fast ≥ 698, worst L2 ≥ 1.01; 20 picked, 20 with the other fish ≥ 10 cm


2-fish, velocity_cm_s: fast ≥ 20, worst L2 ≥ 0.71; 20 picked, 20 with the other fish ≥ 10 cm
2-fish, tail_angular_deg_s: fast ≥ 347, worst L2 ≥ 0.71; 20 picked, 20 with the other fish ≥ 10 cm


## Make the figures

In [3]:
for (name, measure), sel in selections.items():
    c = recs[name]
    what, unit = KINDS[measure]
    out = f"{c['res']}/examples_fast_motion/{measure.rsplit('_', 2)[0]}"
    os.makedirs(out, exist_ok=True)
    rows = []
    for n, (i, r) in enumerate(sel.iterrows(), 1):
        mine_pick = int(c['assigned'].loc[i, f'fish | {ex.MINE}']); den_pick = int(c['assigned'].loc[i, f'fish | {ex.DENIS}'])
        outcome = ex.OUTCOMES[2 * (mine_pick != c['emitter']) + (den_pick != c['emitter'])]
        head = (f"{name} (better list) | {what}: {r[measure]:.0f} {unit} | L2 mine {r.l2_mine:.2f}, Denis {r.l2_denis:.2f} | "
                f"{outcome} | nearest fish {r.gap_cm:.1f} cm")
        path = f"{out}/{n:02d}_frame{int(r.Frame)}.png"
        ex.plot_example(c, i, head, path)
        plt.close('all')
        rows.append(dict(row=i, Frame=int(r.Frame), **{k: r[k] for k in ['velocity_cm_s', 'head_angular_deg_s', 'body_angular_deg_s',
                    'tail_angular_deg_s', 'gap_cm', 'l2_mine', 'l2_denis']}, my_pick=mine_pick, denis_pick=den_pick, outcome=outcome,
                         figure=os.path.basename(path)))
    pd.DataFrame(rows).round(3).to_csv(f"{out}/selection.csv", index=False)
    print(f"{name}, {measure}: {len(rows)} figures in {out}")

3-fish, velocity_cm_s: 20 figures in results/eod_ground_truth_3fish/conductivities/examples_fast_motion/velocity


3-fish, tail_angular_deg_s: 20 figures in results/eod_ground_truth_3fish/conductivities/examples_fast_motion/tail_angular


2-fish, velocity_cm_s: 20 figures in results/eod_ground_truth/conductivities/examples_fast_motion/velocity


2-fish, tail_angular_deg_s: 20 figures in results/eod_ground_truth/conductivities/examples_fast_motion/tail_angular


## Overview of the picked EODs

In [4]:
def flipped_better(c, rows):
    '''How many of the EOD rows the emitter's model fits better with its sign flipped (my model, max |v| = 1).'''
    n = 0
    for i in rows:
        r, U = ex.my_patterns(c, i); u = U[c['emitter']]
        n += np.linalg.norm(u + r) < np.linalg.norm(u - r)
    return n

summary = pd.concat({k: pd.DataFrame({'n': [len(s)], 'other fish ≥ 10 cm': [s.far.sum()],
                                      'mine fits better sign-flipped': [flipped_better(recs[k[0]], s.index)],
                                      'median speed (cm/s)': [s.velocity_cm_s.median()],
                                      'median tail speed (°/s)': [s.tail_angular_deg_s.median()],
                                      'median L2 mine': [s.l2_mine.median()], 'median L2 Denis': [s.l2_denis.median()],
                                      'mine wrong': [(s.index.map(lambda i: int(recs[k[0]]['assigned'].loc[i, f'fish | {ex.MINE}'])) != recs[k[0]]['emitter']).sum()],
                                      'Denis wrong': [(s.index.map(lambda i: int(recs[k[0]]['assigned'].loc[i, f'fish | {ex.DENIS}'])) != recs[k[0]]['emitter']).sum()]})
                     for k, s in selections.items()}, names=['recording', 'measure']).droplevel(-1).round(2)
summary

n  other fish ≥ 10 cm  \
recording measure                                      
3-fish    velocity_cm_s       20                  20   
          tail_angular_deg_s  20                  20   
2-fish    velocity_cm_s       20                  20   
          tail_angular_deg_s  20                  20   

                              mine fits better sign-flipped  \
recording measure                                             
3-fish    velocity_cm_s                                   7   
          tail_angular_deg_s                              0   
2-fish    velocity_cm_s                                  18   
          tail_angular_deg_s                             10   

                              median speed (cm/s)  median tail speed (°/s)  \
recording measure                                                            
3-fish    velocity_cm_s                     33.11                   931.01   
          tail_angular_deg_s                10.92                   814.41   
2-fish    velocity_cm_s                     24.71                   296.46   
          tail_angular_deg_s                13.15                   458.94   

                              median L2 mine  median L2 Denis  mine wrong  \
recording measure                                                           
3-fish    velocity_cm_s                 1.79             2.00          10   
          tail_angular_deg_s            1.46             1.74           7   
2-fish    velocity_cm_s                 2.45             2.43          17   
          tail_angular_deg_s            1.94             2.11           9   

                              Denis wrong  
recording measure                          
3-fish    velocity_cm_s                14  
          tail_angular_deg_s           10  
2-fish    velocity_cm_s                19  
          tail_angular_deg_s           10